# Neural ODE: recovering frequency and phase from the contraction signal

**Question.** Seeing only the contraction signal $a(t)$, can an encoder → latent ODE → decoder model do three things?

1. Learn the hidden dynamics.
2. Infer each trajectory's own frequency $\omega$ and phase $\varphi$.
3. Do both for trajectories it has never seen.

The random search in `cell_simulator_MC.ipynb` fits $\omega$ and $\varphi$ by *assuming* the harmonic oscillator. The Neural ODE must *discover* the dynamics and infer the parameters itself. This is the prerequisite for using latent SDEs on real contraction data, where the form of the dynamics is unknown.

**Setup.**
- **Data:** noise-free, with $\omega \sim |\mathcal N(0.6, 0.15^2)|$ and $\varphi \sim \mathcal N(0, 1.5^2)$, the random-search priors.
- **Latent space:** 2D, the minimum for an oscillation. With two parameters per trajectory, the expected layout is nested rings: phase is the angle, frequency sets which ring.
- **Decoders:** two, compared side by side.
  - `mlp` is an affine map plus an MLP correction whose output layer starts at zero. It is as expressive as a plain MLP, but training starts from an affine decoder. A plain randomly initialised MLP decoder tended to find a degenerate solution: a non-repeating latent path that the decoder folds into oscillations.
  - `linear` keeps only the affine map. It restricts the latent space to affine images of the dynamics.

Sections:
1. Ground truth
2. Training
3. Quantitative evaluation
4. Trajectory fits
5. Latent orbits
6. Parameter recovery
7. The encoder's coordinate grid
8. Next experiments

## 0. Setup
All experiment parameters are in the second cell below.

In [ ]:
# pip install -e .
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from cell_simulator.visualize import COLORS, set_style
from neural_ode.data import make_splits, parameter_grid
from neural_ode.evaluate import (
    encode,
    predict,
)
from neural_ode.evaluate_orbit import (
    fit_metrics,
    fit_phase_per_omega,
    initial_phases,
    latent_diagnostics,
    orbit_coordinates,
)
from neural_ode.model import LatentODE
from neural_ode.system import HarmonicOscillator
from neural_ode.train import TrainConfig, TrainHistory, train
from neural_ode.visualize import (
    plot_encoder_grid,
    plot_latent_orbits,
    plot_parameter_recovery,
    plot_training_history,
    plot_trajectory_fit,
)

set_style()

In [ ]:
SYSTEM = HarmonicOscillator()  # random-search priors for omega and phase
TIMES = np.arange(0.0, 30.0 + 1e-9, 0.5)  # s, 0.5 s per frame
N_TRAIN, N_VAL, N_TEST = 128, 32, 32
DATA_SEED = 0  # fixed: seeds below vary model init and batch order only

LATENT_DIM = 2  # 3 or more switches the orbit and encoder plots to 3D views
DECODERS = ["mlp"]
SEEDS = [0, 1, 2]
N_ITERS = 4000
RETRAIN = False  # False: load checkpoints when they exist

PROBE_OMEGAS = np.linspace(0.35, 0.85, 6)  # grid of unseen parameters for sections 3, 6, 7
N_PROBE_PHASES = 12
SHOW_SEED = 0  # seed shown in the per-model plots

MODEL_ROOT = Path(f"../models/neural_ode")
FIG_DIR = Path(f"../reports/figures/neural_ode")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Ground truth from `cell_simulator`

Every trajectory has its own $(\omega, \varphi)$, drawn from the random-search priors. Its hidden state $(c, z)$ runs round the unit circle at speed $\omega$, starting at the point set by $\varphi$. So all trajectories share one circle in the true state space and differ in speed and starting point. The model sees only $a(t)$.

In [ ]:
splits = make_splits(SYSTEM, TIMES, N_TRAIN, N_VAL, N_TEST, seed=DATA_SEED)
train_data, val_data, test_data = splits["train"], splits["val"], splits["test"]
probe_phases = np.linspace(0, 2 * np.pi, N_PROBE_PHASES, endpoint=False)
probe = parameter_grid(SYSTEM, TIMES, {"omega": PROBE_OMEGAS, "phase": probe_phases})
print({k: len(v) for k, v in splits.items()}, "| probe grid:", len(probe))
periods = SYSTEM.periods(train_data.params)
print(f"periods: {periods.min():.1f}-{periods.max():.1f} s (window {TIMES[-1]:.0f} s)")

fig, (ax_p, ax_a) = plt.subplots(1, 2, figsize=(12, 3.8), layout="constrained", width_ratios=[1, 1.8])
for k in np.argsort(train_data.params["omega"])[[0, 40, 90, -1]]:
    ax_a.plot(
        TIMES,
        train_data.observations[k, :, 0],
        label=f"ω = {train_data.params['omega'][k]:.2f}, φ = {train_data.params['phase'][k]:.2f}",
    )
ax_a.set(xlabel="time [s]", ylabel="a(t)", title="observed: contraction signal")
ax_a.legend(frameon=False, fontsize=8, loc="lower right")
for name, d, marker in [("train", train_data, "o"), ("val", val_data, "s"), ("test", test_data, "^")]:
    ax_p.scatter(d.params["omega"], d.params["phase"], s=12, marker=marker, label=name, alpha=0.8)
ax_p.scatter(probe.params["omega"], probe.params["phase"], s=6, color="0.6", label="probe grid")
ax_p.set(xlabel="ω [rad/s]", ylabel="φ [rad]", title="parameters")
ax_p.legend(frameon=False, fontsize=8, loc="upper right")
fig.savefig(FIG_DIR / "ground_truth.png", bbox_inches="tight")
plt.show()

## 2. Training

The loss is the trajectory MSE of the reconstructed $a(t)$, optimised with Adam, gradient clipping and cosine learning-rate decay, keeping the best-validation model.

One model is trained per decoder and seed. Each takes several minutes on a CPU. Checkpoints are saved under `MODEL_ROOT` and reused unless `RETRAIN = True`.

In [ ]:
def fit_or_load(decoder: str, seed: int) -> tuple[LatentODE, TrainHistory]:
    model_dir = MODEL_ROOT / f"{decoder}_decoder_latent_dim_{LATENT_DIM}"
    model_dir.mkdir(parents=True, exist_ok=True)
    path = model_dir / f"seed_{seed}.pt"
    torch.manual_seed(seed)
    model = LatentODE(latent_dim=LATENT_DIM, linear_decoder=decoder == "linear")
    if path.exists() and not RETRAIN:
        checkpoint = torch.load(path)
        model.load_state_dict(checkpoint["model"])
        return model, TrainHistory(**checkpoint["history"])
    history = train(model, train_data, val_data, TrainConfig(n_iters=N_ITERS, seed=seed))
    torch.save(
        {
            "model": model.state_dict(),
            "history": asdict(history),
            "latent_dim": LATENT_DIM,
            "linear_decoder": decoder == "linear",
            "n_iters": N_ITERS,
        },
        path,
    )
    return model, history


models, histories = {}, {}
for decoder in DECODERS:
    for seed in SEEDS:
        models[decoder, seed], histories[decoder, seed] = fit_or_load(decoder, seed)
        print(f"{decoder:6s} seed {seed}: best val MSE {min(histories[decoder, seed].val_loss):.2e}")

In [ ]:
fig_dir = FIG_DIR / f"{decoder}_decoder_latent_dim_{LATENT_DIM}"
fig_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(
    len(DECODERS),
    len(SEEDS),
    figsize=(4.3 * len(SEEDS), 3.2 * len(DECODERS)),
    layout="constrained",
    sharey=True,
    squeeze=False,
)
for i, decoder in enumerate(DECODERS):
    for j, seed in enumerate(SEEDS):
        plot_training_history(histories[decoder, seed], ax=axes[i, j])
        axes[i, j].set_title(f"{decoder} decoder, seed {seed}")
fig.savefig(fig_dir / "training_history.png", bbox_inches="tight")
plt.show()

### Check all dimensions

In [ ]:
times, observations = train_data.tensors()
batch_size = min(TrainConfig().batch_size, len(train_data))
target = observations[:batch_size]

rows = [
    ("data", "times", tuple(times.shape), "T"),
    ("data", "train observations", tuple(observations.shape), "N, T, obs_dim"),
    ("input", "target batch", tuple(target.shape), "B, T, obs_dim"),
]

for decoder in DECODERS:
    seed = 0
    model = models[decoder, seed].eval()
    with torch.no_grad():
        z0 = model.encoder(target)
        drift_at_z0 = model.drift(times[0], z0)
        latents = model.integrate(z0, times)
        pred = model.decoder(latents)
    rows.extend(
        [
            (f"{decoder}, seed {seed}", "encoder z0 (pre-drift)", tuple(z0.shape), "B, latent_dim"),
            (f"{decoder}, seed {seed}", "drift result f(z0)", tuple(drift_at_z0.shape), "B, latent_dim"),
            (f"{decoder}, seed {seed}", "integrated z(t) (latents)", tuple(latents.shape), "B, T, latent_dim"),
            (f"{decoder}, seed {seed}", "prediction", tuple(pred.shape), "B, T, obs_dim"),
        ]
    )

widths = (18, 25, 18, 22)
header = ("model", "tensor or stage", "shape", "axis order")
print(" | ".join(f"{value:<{width}}" for value, width in zip(header, widths)))
print("-+-".join("-" * width for width in widths))
for row in rows:
    print(" | ".join(f"{str(value):<{width}}" for value, width in zip(row, widths)))

## 3. Quantitative evaluation

There are two test sets:
- **Held-out trajectories**, with i.i.d. parameters, for fit and dynamics.
- **The probe grid** of unseen $(\omega, \varphi)$, for phase.

The encoder reads the whole trajectory, so this is reconstruction of unseen trajectories, not forecasting.

How to read the columns:

- **Output fit.** `nMSE` is the MSE of $a(t)$ divided by its variance: 1 means "no better than the mean", 0 is perfect.
- **Parameter recovery.** Both measures are deformation-invariant, so they measure what the data actually determine.
  - `ω err`: median relative error of the winding frequency $\hat\omega$ of each latent orbit.
  - `ω̂ R²`: $\hat\omega$ against the true $\omega$ on the identity line.
  - `φ RMS`: phase error of the probe trajectories in rad, after allowing each $\omega$ its own rotation. Random guessing gives about 1.8.

Output fit alone is not enough: a model can reproduce $a(t)$ with a latent space unrelated to the true dynamics.

In [ ]:
predictions, diagnostics, probe_predictions, rows = {}, {}, {}, []
for (decoder, seed), model in models.items():
    pred = predictions[decoder, seed] = predict(model, test_data)
    diag = diagnostics[decoder, seed] = latent_diagnostics(pred, test_data)
    probe_pred = probe_predictions[decoder, seed] = predict(model, probe)
    coords = orbit_coordinates(probe_pred.latents, diag.frame)
    phase_fits = fit_phase_per_omega(
        initial_phases(coords(probe_pred.latents)), probe.params["phase"], probe.params["omega"]
    )
    s = diag.summary()
    rows.append(
        {
            "decoder": decoder,
            "seed": seed,
            "nMSE": fit_metrics(pred.observations, test_data.observations).nmse,
            "ω err": s["omega_rel_error"],
            "ω̂ R²": s["omega_hat_r2"],
            "φ RMS": float(np.sqrt(np.mean([f.rms_error**2 for f in phase_fits.values()]))),
        }
    )

keys = [k for k in rows[0] if k not in ("decoder", "seed")]
print("decoder seed" + "".join(f"{k:>12s}" for k in keys))
for decoder in DECODERS:
    sub = [r for r in rows if r["decoder"] == decoder]
    for r in sub:
        print(f"{decoder:7s}{r['seed']:5d}" + "".join(f"{r[k]:12.3g}" for k in keys))
    print(f"{decoder:7s} med" + "".join(f"{np.median([r[k] for r in sub]):12.3g}" for k in keys) + "\n")

## 4. Trajectory fits

For `SHOW_SEED`: the best and worst held-out trajectories and two in between, plus the per-frame error over all test trajectories.

In [ ]:
for decoder in DECODERS:
    fig = plot_trajectory_fit(test_data, predictions[decoder, SHOW_SEED])
    fig.suptitle(f"{decoder} decoder, seed {SHOW_SEED}")
    fig.savefig(fig_dir / f"trajectory_fit_seed_{SHOW_SEED}.png", bbox_inches="tight")
    plt.show()

In [ ]:
import matplotlib as mpl
from scipy.ndimage import uniform_filter1d

SMOOTH = 10  # frames (5 s), smooths the per-trajectory curves
LATE_FROM = 25.0  # s, start of the window where the mean error rises

for decoder in DECODERS:
    key = (decoder, SHOW_SEED)
    err = (np.asarray(predictions[key].observations) - np.asarray(test_data.observations))[
        ..., 0
    ] ** 2  # (n_traj, n_times)
    omega = test_data.params["omega"]
    cmap, norm = plt.get_cmap("viridis"), mpl.colors.Normalize(omega.min(), omega.max())

    fig, (ax_a, ax_b, ax_c) = plt.subplots(1, 3, figsize=(16, 4), layout="constrained")

    # (a) mean vs median: a flat median under a rising mean means a few outliers drive the rise
    ax_a.semilogy(TIMES, err.mean(0), label="mean")
    ax_a.semilogy(TIMES, np.median(err, 0), label="median")
    ax_a.set(xlabel="time [s]", ylabel="squared error", title="(a) mean vs median over trajectories")
    ax_a.legend(frameon=False)

    # (b) per-trajectory error, smoothed (the raw error drops to ~0 whenever the curves cross)
    smooth = uniform_filter1d(err, SMOOTH, axis=1, mode="nearest")
    for i in np.argsort(omega):
        ax_b.semilogy(TIMES, smooth[i], color=cmap(norm(omega[i])), lw=0.9)
    ax_b.set(xlabel="time [s]", title=f"(b) per trajectory, {SMOOTH * (TIMES[1] - TIMES[0]):.0f} s running mean")
    fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax_b, label="true ω [rad/s]")

    # (c) early vs late error against omega: late errors only at the edges points to data coverage
    late = TIMES >= LATE_FROM
    ax_c.semilogy(omega, err[:, ~late].mean(1), "o", color="0.6", ms=4, label=f"t < {LATE_FROM:.0f} s")
    ax_c.semilogy(omega, err[:, late].mean(1), "o", ms=4, label=f"t ≥ {LATE_FROM:.0f} s")
    ax_c.axvline(SYSTEM.omega_mean, color="k", lw=0.8, ls="--")
    ax_c.set(xlabel="true ω [rad/s]", ylabel="mean squared error", title="(c) error vs ω (dashed: prior mean)")
    ax_c.legend(frameon=False)

    fig.suptitle(f"{decoder} decoder, seed {SHOW_SEED}")
    fig.savefig(fig_dir / f"error_breakdown_seed_{SHOW_SEED}.png", bbox_inches="tight")
    plt.show()

## 5. Latent orbits

The learned family of orbits, coloured by true $\omega$, drawn over streamlines of the learned vector field $f_\theta$.

The latent space is only identifiable up to a smooth deformation, so look at topology rather than exact shape. The ideal picture is one closed orbit per $\omega$, nested and ordered by $\omega$. Whether fast orbits sit inside or outside is a convention. Warning signs are orbits that do not close, and long sweeps that the decoder folds into oscillations.

The streamlines show what $f_\theta$ does *between* the rings, where there are no data. With `LATENT_DIM` ≥ 3 the figure switches to 3D, top and side views.

In [ ]:
for decoder in DECODERS:
    key = (decoder, SHOW_SEED)
    fig = plot_latent_orbits(test_data, predictions[key], diagnostics[key], models[key])
    fig.suptitle(f"{decoder} decoder, seed {SHOW_SEED}")
    fig.savefig(fig_dir / f"latent_orbits_seed_{SHOW_SEED}.png", bbox_inches="tight")
    plt.show()

## 6. Parameter recovery

This is the Neural ODE counterpart of the random search. Neither parameter is an explicit output of the model, so both are read out afterwards.

**(a) ω.** $\hat\omega$ is how fast each latent orbit winds. Time is not deformed, so this is directly comparable to the true $\omega$.

**(b) φ.** The angle of $z_0$ on its own orbit, for the probe grid of unseen trajectories. The latent phase is only determined up to a rotation per $\omega$ (and a reflection), so each $\omega$ level gets its own offset.

Points on the diagonal mean the parameter is recovered.

In [ ]:
for decoder in DECODERS:
    key = (decoder, SHOW_SEED)
    fig = plot_parameter_recovery(diagnostics[key], probe, probe_predictions[key])
    fig.suptitle(f"{decoder} decoder, seed {SHOW_SEED}")
    fig.savefig(fig_dir / f"parameter_recovery_seed_{SHOW_SEED}.png", bbox_inches="tight")
    plt.show()

## 7. The encoder's coordinate grid

Where does the encoder put $z_0$ for a regular grid of unseen $(\omega, \varphi)$?
- Solid loops join equal $\omega$.
- Dashed lines join equal $\varphi$.

A clean grid, with rings nested by $\omega$ and dashed lines that never cross, means the encoder maps parameter space one-to-one and smoothly. New trajectories then get their $(\omega, \varphi)$ from a single encoder pass: amortised inference, instead of a random search per trajectory. Crossing or collapsing lines mean the encoder confuses parameters.

In [ ]:
for decoder in DECODERS:
    key = (decoder, SHOW_SEED)
    z0 = encode(models[key], probe)
    fig = plot_encoder_grid(probe, z0, diagnostics[key].frame, N_PROBE_PHASES)
    fig.axes[0].set_aspect("equal", adjustable="box")
    fig.suptitle(f"{decoder} decoder, seed {SHOW_SEED}", y=0.93)
    fig.savefig(fig_dir / f"encoder_grid_seed_{SHOW_SEED}.png", bbox_inches="tight")
    plt.show()

In [ ]:
import matplotlib as mpl

key = ("mlp", SHOW_SEED)
probe_pred = probe_predictions[key]

# Same coordinates as the encoder-grid plot
coords = orbit_coordinates(probe_pred.latents, diagnostics[key].frame)
Z = np.asarray(probe_pred.latents)[..., :2]  # (n_trajectories, n_times, 2), radii preserved
assert Z.ndim == 3 and Z.shape[1] == len(TIMES), Z.shape

# Group the probe trajectories into rings (equal omega), sorted by phase
omega_p, phase_p = probe.params["omega"], probe.params["phase"]
omega_levels = np.unique(omega_p)
rings = [np.where(omega_p == w)[0][np.argsort(phase_p[omega_p == w])] for w in omega_levels]

cmap = plt.get_cmap("viridis")
norm = mpl.colors.Normalize(omega_levels.min(), omega_levels.max())
pad = 0.1 * np.abs(Z).max()
xlim = (Z[..., 0].min() - pad, Z[..., 0].max() + pad)
ylim = (Z[..., 1].min() - pad, Z[..., 1].max() + pad)


def draw_grid(ax, t_idx):
    for w, idx in zip(omega_levels, rings):
        pts = Z[idx, t_idx]
        pts = np.vstack([pts, pts[:1]])  # close the ring
        ax.plot(pts[:, 0], pts[:, 1], color=cmap(norm(w)), lw=1.5)
    for k in range(len(rings[0])):  # lines of equal phase across rings
        pts = Z[[r[k] for r in rings], t_idx]
        ax.plot(pts[:, 0], pts[:, 1], ls="--", color="0.6", lw=0.7)
    zero = [r[0] for r in rings]  # phase 0 on each ring
    ax.scatter(Z[zero, t_idx, 0], Z[zero, t_idx, 1], color="k", s=15, zorder=3)
    ax.set(xlim=xlim, ylim=ylim, aspect="equal", title=f"t = {TIMES[t_idx]:.1f} s")


snapshot_times = [0.0, 2.5, 5.0, 10.0]
fig, axes = plt.subplots(1, len(snapshot_times), figsize=(4 * len(snapshot_times), 4.3), layout="constrained")
for ax, t in zip(axes, snapshot_times):
    draw_grid(ax, int(np.searchsorted(TIMES, t)))
axes[0].set(xlabel="latent dim 1", ylabel="latent dim 2")
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, label="true ω [rad/s]", shrink=0.8)
fig.suptitle(f"Probe grid moved by the learned dynamics ({key[0]} decoder, seed {key[1]})")
fig.savefig(fig_dir / f"latent_flow_seed_{key[1]}.png", bbox_inches="tight")
plt.show()

In [ ]:
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

fig, ax = plt.subplots(figsize=(5.5, 5))


def update(i):
    ax.clear()
    draw_grid(ax, i)
    return []


anim = FuncAnimation(fig, update, frames=len(TIMES), interval=150)

fps = 1000 / 150  # same speed as the notebook animation

anim.save(fig_dir / f"encoder_grid_flow_seed_{key[1]}.gif", writer="pillow", fps=fps, dpi=100)

plt.close(fig)
HTML(anim.to_jshtml())

# Test questions

In [ ]:
# Cells for neural_ode.ipynb, answering Bjørn's three questions.
# Paste each "# %%" block as its own cell at the end of the notebook (after the
# probe predictions exist). Assumes LATENT_DIM == 2 and the notebook's variables:
# models, DECODERS, SEEDS, SHOW_SEED, test_data, probe, probe_predictions, TIMES, FIG_DIR.

# %% Q1: the latent space is not unique (scale + rotation leave a(t) unchanged)
import matplotlib as mpl
import torch
from scipy.stats import spearmanr

from neural_ode.solvers import odeint_rk4

assert LATENT_DIM == 2


class Reparam:
    """The same model in new latent coordinates z' = A z, with A = scale * rotation.

    Encoder:  z0' = A z0
    Drift:    f'(z') = A f(A^-1 z')     (since dz'/dt = A dz/dt)
    Decoder:  g'(z') = g(A^-1 z')
    """

    def __init__(self, model, scale=2.0, angle=np.pi / 3):
        dtype = next(model.parameters()).dtype
        c, s = np.cos(angle), np.sin(angle)
        self.A = scale * torch.tensor([[c, -s], [s, c]], dtype=dtype)
        self.A_inv = torch.linalg.inv(self.A)
        self.model, self.steps_per_frame = model, model.steps_per_frame

    def encoder(self, obs):
        return self.model.encoder(obs) @ self.A.T

    def drift(self, t, z):
        return self.model.drift(t, z @ self.A_inv.T) @ self.A.T

    def decoder(self, z):
        return self.model.decoder(z @ self.A_inv.T)


def run(m, data):
    """Encode, integrate and decode; returns a_hat (n, T, 1) and latents (n, T, 2)."""
    dtype = m.A.dtype if isinstance(m, Reparam) else next(m.parameters()).dtype
    obs = torch.as_tensor(data.observations, dtype=dtype)
    t = torch.as_tensor(TIMES, dtype=dtype)
    with torch.no_grad():
        z = odeint_rk4(m.drift, m.encoder(obs), t, m.steps_per_frame).transpose(0, 1)
        return m.decoder(z).numpy(), z.numpy()


def features(z):
    """Per trajectory: mean |z| (Bjørn's 'size of the coordinates') and winding frequency omega_hat."""
    d = z - z.reshape(-1, 2).mean(0)  # centre of all orbits
    angle = np.unwrap(np.arctan2(d[..., 1], d[..., 0]), axis=1)
    omega_hat = np.abs(np.polyfit(TIMES, angle.T, 1)[0])  # slope of angle vs time
    return np.linalg.norm(z, axis=-1).mean(1), omega_hat


model = models["mlp", SHOW_SEED]
a1, z1 = run(model, test_data)
a2, z2 = run(Reparam(model), test_data)
r1, w1 = features(z1)
r2, w2 = features(z2)
print(f"max |a(t) difference|      : {np.abs(a1 - a2).max():.1e}   (same function)")
print(f"mean radius, new / old     : {np.median(r2 / r1):.2f}      (changes with the coordinates)")
print(f"max |omega_hat difference| : {np.abs(w1 - w2).max():.1e}   (invariant)")

omega = test_data.params["omega"]
cmap, norm = plt.get_cmap("viridis"), mpl.colors.Normalize(omega.min(), omega.max())
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), layout="constrained")
for ax, z, title in [(axes[0], z1, "trained model"), (axes[1], z2, "z' = 2 R(60°) z, same a(t)")]:
    for i in np.argsort(omega):
        ax.plot(z[i, :, 0], z[i, :, 1], color=cmap(norm(omega[i])), lw=0.8)
    ax.set(aspect="equal", title=title, xlabel="$z_1$", ylabel="$z_2$")
fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), ax=axes, label="true ω [rad/s]", shrink=0.8)
fig.savefig(fig_dir / "q1_reparametrisation.png", bbox_inches="tight")
plt.show()

# %% Q2: can we tell two cells with different frequency apart, across all trained models?
fig, (ax_r, ax_w) = plt.subplots(1, 2, figsize=(11, 4.2), layout="constrained")
print(f"{'model':14s}{'ρ(mean |z|, ω)':>18s}{'ρ(ω̂, ω)':>12s}")
for decoder, marker in zip(DECODERS, ["o", "^"]):
    for seed in SEEDS:
        r, w = features(run(models[decoder, seed], test_data)[1])
        rho_r, rho_w = spearmanr(r, omega)[0], spearmanr(w, omega)[0]
        print(f"{decoder:7s}seed {seed}{rho_r:+18.2f}{rho_w:+12.2f}")
        label = f"{decoder}, seed {seed}"
        ax_r.plot(omega, r, marker, ms=4, label=label)
        ax_w.plot(omega, w, marker, ms=4, label=label)
lims = [omega.min(), omega.max()]
ax_w.plot(lims, lims, "k--", lw=0.8)
ax_r.set(xlabel="true ω [rad/s]", ylabel="mean |z| over the window", title="(a) size of the coordinates")
ax_w.set(xlabel="true ω [rad/s]", ylabel="ω̂ [rad/s]", title="(b) winding frequency (dashed: identity)")
ax_r.legend(frameon=False, fontsize=8)
fig.savefig(fig_dir / "q2_discrimination.png", bbox_inches="tight")
plt.show()

# %% Q3: latent orbits annotated with time
T_SHOW, MARK_EVERY = 7.5, 1.5  # s: window shown and spacing of the time markers
key = ("mlp", SHOW_SEED)
Zp = np.asarray(probe_predictions[key].latents)[..., :2]
omega_p, phase_p = probe.params["omega"], probe.params["phase"]
levels = np.unique(omega_p)
starts = [np.where((omega_p == w) & np.isclose(phase_p, 0.0))[0][0] for w in levels]  # phase 0 on each ring
shown = TIMES <= T_SHOW + 1e-9
marks = np.where(shown & np.isclose(np.mod(TIMES, MARK_EVERY), 0.0))[0]
cmap_p, norm_p = plt.get_cmap("viridis"), mpl.colors.Normalize(levels.min(), levels.max())

fig, (ax_o, ax_a) = plt.subplots(1, 2, figsize=(12, 5), layout="constrained", width_ratios=[1.1, 1])
for k in marks:  # isochrones: points reached at the same time, starting from the same phase
    ax_o.plot(Zp[starts, k, 0], Zp[starts, k, 1], color="0.75", lw=0.8, zorder=1)
for w, i in zip(levels, starts):
    c = cmap_p(norm_p(w))
    ax_o.plot(Zp[i, shown, 0], Zp[i, shown, 1], color=c, lw=1.2)
    ax_o.plot(Zp[i, marks, 0], Zp[i, marks, 1], "o", color=c, ms=4, zorder=3)
for i in (starts[0], starts[-1]):  # label the outermost and innermost ring
    for k in marks:
        ax_o.annotate(f"{TIMES[k]:g}", Zp[i, k], xytext=(4, 4), textcoords="offset points", fontsize=7)
ax_o.set(
    aspect="equal",
    xlabel="$z_1$",
    ylabel="$z_2$",
    title=f"Orbits from phase 0, markers every {MARK_EVERY:g} s (grey: same time)",
)

centre = Zp.reshape(-1, 2).mean(0)
for w, i in zip(levels, starts):
    d = Zp[i] - centre
    angle = np.abs(np.unwrap(np.arctan2(d[:, 1], d[:, 0])) - np.arctan2(d[0, 1], d[0, 0]))
    ax_a.plot(TIMES, angle, color=cmap_p(norm_p(w)), lw=1.2)
    ax_a.plot(TIMES, w * TIMES, color=cmap_p(norm_p(w)), lw=0.8, ls="--")
ax_a.set(xlabel="time [s]", ylabel="angle travelled [rad]", title="(b) angle vs time (dashed: true ω·t)")
fig.colorbar(mpl.cm.ScalarMappable(norm=norm_p, cmap=cmap_p), ax=[ax_o, ax_a], label="true ω [rad/s]", shrink=0.8)
fig.savefig(fig_dir / f"q3_time_annotated_seed_{key[1]}.png", bbox_inches="tight")
plt.show()

## 8. Check adjoint method

In [ ]:
from neural_ode.adjoint_gradient_check import compare_gradients, plot_adjoint_summary, profile_cost

ADJOINT_KEY = ("mlp", SHOW_SEED)
adjoint_model = models[ADJOINT_KEY]

comparison = compare_gradients(adjoint_model, train_data)
cost = profile_cost(adjoint_model, train_data, steps_per_frame=[1, 2, 4, 8, 16, 32], n_repeat=3)

print(f"loss  backprop {comparison.loss_backprop:.6e}  adjoint {comparison.loss_adjoint:.6e}")
print(f"relative gradient error: {comparison.total_rel_error:.2e}")
print(f"tensors without an adjoint gradient: {comparison.missing or 'none'}")
print(f"\n{'gradient':>10s}{'steps':>8s}{'memory [MB]':>14s}{'forward [s]':>13s}{'backward [s]':>14s}")
for r in cost:
    print(f"{r['gradient']:>10s}{r['n_steps']:8d}{r['saved_mb']:14.2f}{r['forward_s']:13.3f}{r['backward_s']:14.3f}")

fig = plot_adjoint_summary(comparison, cost)
fig.suptitle(f"Adjoint vs backprop, batch of 32 ({ADJOINT_KEY[0]} decoder, seed {ADJOINT_KEY[1]})")
fig.savefig(fig_dir / f"adjoint_summary_seed_{ADJOINT_KEY[1]}.png", bbox_inches="tight")
plt.show()

## 9. Next experiments

- **Decoder choice.** Compare the two decoders in section 3 before choosing one for the latent SDE.
- **Latent dimension.** Set `LATENT_DIM = 3` to compare. In a 3D run, the model also chose tilted nested rings rather than a cylinder.
- **Amplitude as a third parameter.** This requires `RK4.integrate` with arbitrary $x_0$. It is the only way the data can distinguish a centre from a limit cycle.
- **More training data.** The MLP decoder shows a train/validation gap.
- **Random-window training.** Set `TrainConfig(window_frames=...)` to train on random time windows. One trial with the plain MLP decoder did not help.
- **Forecasting.** Encode only the first part of each trajectory and predict the rest.